# 2026 COMP90042 Project
*Make sure you change the file name with your group id.*

# Readme
*If there is something to be noted for the marker, please mention here.*

*If you are planning to implement a program with Object Oriented Programming style, please put those the bottom of this ipynb file*

# 1.DataSet Processing
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

# 2.Model Implementation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

# 3.Testing and Evaluation
(You can add as many code blocks and text blocks as you need. However, YOU SHOULD NOT MODIFY the section title)

## Object Oriented Programming codes here

*You can use multiple code snippets. Just add more if needed*

In [ ]:
from classifier import load_retriever_cache
from collections import Counter

TRAIN_CACHE = "cache/train-retriever-only-k4-bm25200.json"
DEV_CACHE   = "cache/dev-retriever-only-k4-bm25200.json"
TEST_CACHE  = "cache/test-retriever-only-k4-bm25200.json"

train_pipeline = load_retriever_cache(TRAIN_CACHE)
dev_pipeline   = load_retriever_cache(DEV_CACHE)
test_pipeline  = load_retriever_cache(TEST_CACHE)

for name, d in [("Train", train_pipeline), ("Dev", dev_pipeline), ("Test", test_pipeline)]:
    counts = dict(Counter(v["claim_label"] for v in d.values()))
    print(f"{name:5s}: {len(d):4d} claims | {counts}")

In [ ]:
ft_model, ft_tokenizer = train_deberta(
    train_data            = train_pipeline,   # pipeline-retrieved evidence
    dev_data              = dev_pipeline,     # pipeline-retrieved evidence
    evidence_dict         = evidence_dict,
    bm25_retriever        = None,             # NEI uses cached IDs, not runtime BM25
    deberta_best_dir      = DEBERTA_BEST_DIR,
    hf_deberta_repo       = HF_DEBERTA_REPO,
    epochs                = 5,
    batch_size            = 4,
    disputed_weight_boost = 3.0,
    reuse_if_found        = True,
)

In [ ]:
from sklearn.metrics import classification_report
from classifier import eval_macro_f1_dev, LABEL_NAMES

test_f1, y_true, y_pred = eval_macro_f1_dev(
    ft_model, ft_tokenizer,
    test_pipeline, evidence_dict,
    device         = ft_model.device,
    batch_size     = 16,
    bm25_retriever = None,   # use cached test evidence IDs
)
print(f"\nPipeline test macro-F1: {test_f1:.4f}\n")
print(classification_report(y_true, y_pred, labels=LABEL_NAMES, zero_division=0))